In [0]:
# CONFIGURACIÓN DEL CATÁLOGO

catalogo = "silver"

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS rendimiento_estudiantil.silver
""")

In [0]:
bronze_estudiantes = spark.table(f"rendimiento_estudiantil.bronze.estudiantes")
bronze_cursos = spark.table(f"rendimiento_estudiantil.bronze.cursos")
bronze_matriculas = spark.table(f"rendimiento_estudiantil.bronze.matriculas")
bronze_semestres = spark.table(f"rendimiento_estudiantil.bronze.semestres")

Limpiar estudiantes

In [0]:
from pyspark.sql.functions import col, trim

silver_estudiantes = (
    bronze_estudiantes
    .dropDuplicates(["Student_ID"])
    .fillna({
        "Gender":"No especificado",
        "Region_Type":"No especificado",
        "Parent_Education":"No especificado",
        "Family_Income_Level":"No especificado",
        "Home_City":"No especificado",
        "Major_Subject":"No especificado",
        "email":"Sin correo"
    })
    # Limpiar espacios en columnas de texto
    .withColumn("Gender", trim(col("Gender")))
    .withColumn("Region_Type", trim(col("Region_Type")))
    .withColumn("Parent_Education", trim(col("Parent_Education")))
    .withColumn("Family_Income_Level", trim(col("Family_Income_Level")))
    .withColumn("Home_City", trim(col("Home_City")))
    .withColumn("Major_Subject", trim(col("Major_Subject")))
    .withColumn("email", trim(col("email")))
    # Casts y filtros
    .withColumn("Student_ID", col("Student_ID").cast("int"))
    .withColumn("Age", col("Age").cast("int"))
    .filter(col("Age") >= 16)
)

Guardar estudiantes

In [0]:
(
    silver_estudiantes.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"rendimiento_estudiantil.silver.estudiantes")
)

Limpiar cursos

In [0]:
silver_cursos = (
    bronze_cursos
    .dropDuplicates(["id_curso"])
    .fillna({
        "nombre_curso":"Sin nombre",
        "major_subject":"Sin área",
        "nivel":"No definido"
    })
    # Limpiar espacios en columnas de texto
    .withColumn("nombre_curso", trim(col("nombre_curso")))
    .withColumn("major_subject", trim(col("major_subject")))
    .withColumn("nivel", trim(col("nivel")))
    # Casts y filtros
    .withColumn("id_curso", col("id_curso").cast("int"))
    .withColumn("creditos", col("creditos").cast("int"))
    .withColumn("costo_matricula", col("costo_matricula").cast("double"))
    .filter(col("creditos") > 0)
)

In [0]:
(
    silver_cursos.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"rendimiento_estudiantil.silver.cursos")
)

Limpiar matículas

In [0]:
from pyspark.sql.functions import to_date

silver_matriculas = (
    bronze_matriculas
    .dropDuplicates(["id_matricula"])
    .fillna({
        "nota_final":0,
        "estado":"Sin estado"
    })
    # Limpiar espacios en columnas de texto
    .withColumn("estado", trim(col("estado")))
    # Casts y transformaciones
    .withColumn("Student_ID", col("Student_ID").cast("int"))
    .withColumn("Semester_ID", col("Semester_ID").cast("int"))
    .withColumn("id_curso", col("id_curso").cast("int"))
    .withColumn("nota_final", col("nota_final").cast("double"))
    .withColumn("costo_matricula", col("costo_matricula").cast("double"))
    .withColumn("fecha_matricula", to_date("fecha_matricula","M/d/yyyy"))
    .filter(
        (col("nota_final") >= 0) &
        (col("nota_final") <= 100)
    )
)

In [0]:
(
    silver_matriculas.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"rendimiento_estudiantil.silver.matriculas")
)

Limpiar semestres

In [0]:
silver_semestres = (
    bronze_semestres
    .dropDuplicates(["Semester_ID"])
    .fillna({
        "modalidad":"No definida"
    })
    # Limpiar espacios en columnas de texto
    .withColumn("periodo", trim(col("periodo")))
    .withColumn("ciclo", trim(col("ciclo")))
    .withColumn("modalidad", trim(col("modalidad")))
    # Casts
    .withColumn("Semester_ID", col("Semester_ID").cast("int"))
    .withColumn("anio", col("anio").cast("int"))
)

In [0]:
(
    silver_semestres.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"rendimiento_estudiantil.silver.semestres")
)

Verificar silver

In [0]:
%sql
-- Verificar tablas silver
SHOW TABLES IN rendimiento_estudiantil.silver